# 01 — Data Collection

**World Cup 2030 Prediction Project**

This notebook downloads the real, source-verified datasets described in `data/DATA_SOURCES.md`.
No data is generated, simulated, or fabricated here — every row comes from a live HTTP request
to a public, documented source.


In [1]:
import pandas as pd
import numpy as np
import urllib.request
import os

RAW_DIR = "../data/raw"
os.makedirs(RAW_DIR, exist_ok=True)

SOURCES = {
    "results.csv": "https://raw.githubusercontent.com/martj42/international_results/master/results.csv",
    "fifa_ranking.csv": "https://raw.githubusercontent.com/cnc8/fifa-world-ranking/master/fifa_ranking-2020-12-10.csv",
}

for filename, url in SOURCES.items():
    dest = os.path.join(RAW_DIR, filename)
    print(f"Downloading {filename} from {url}")
    urllib.request.urlretrieve(url, dest)
    size_kb = os.path.getsize(dest) / 1024
    print(f"  -> saved to {dest} ({size_kb:,.1f} KB)")


  -> saved to ../data/raw/results.csv (3,640.5 KB)


  -> saved to ../data/raw/fifa_ranking.csv (2,991.4 KB)


## Quick sanity check on what we just downloaded

In [2]:
matches = pd.read_csv(os.path.join(RAW_DIR, "results.csv"))
ranking = pd.read_csv(os.path.join(RAW_DIR, "fifa_ranking.csv"))

print("MATCHES:", matches.shape)
print("RANKING:", ranking.shape)
matches.head()


MATCHES: (49520, 9)
RANKING: (62424, 9)


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,1872-11-30,Scotland,England,0,0,Friendly,Glasgow,Scotland,False
1,1873-03-08,England,Scotland,4,2,Friendly,London,England,False
2,1874-03-07,Scotland,England,2,1,Friendly,Glasgow,Scotland,False
3,1875-03-06,England,Scotland,2,2,Friendly,London,England,False
4,1876-03-04,Scotland,England,3,0,Friendly,Glasgow,Scotland,False


In [3]:
ranking.head()


,id,rank,country_full,country_abrv,total_points,previous_points,rank_change,confederation,rank_date
0,43885,104,Swaziland,SWZ,10,0,0,CAF,1992-12-31
1,43972,42,Turkey,TUR,31,0,0,UEFA,1992-12-31
2,43952,43,Northern Ireland,NIR,31,0,0,UEFA,1992-12-31
3,43945,44,Finland,FIN,31,0,0,UEFA,1992-12-31
4,43976,45,Australia,AUS,29,0,0,AFC,1992-12-31


## Provenance log

Recording exactly what was fetched, when, and how many rows — this is our audit trail so the
project is reproducible and every number in the final report can be traced back to a source.


In [4]:
import datetime

log = pd.DataFrame([
    {"file": "results.csv", "source_url": SOURCES["results.csv"],
     "rows": len(matches), "cols": matches.shape[1],
     "retrieved_at": datetime.datetime.now().isoformat()},
    {"file": "fifa_ranking.csv", "source_url": SOURCES["fifa_ranking.csv"],
     "rows": len(ranking), "cols": ranking.shape[1],
     "retrieved_at": datetime.datetime.now().isoformat()},
])
log.to_csv(os.path.join(RAW_DIR, "_provenance_log.csv"), index=False)
log


,file,source_url,rows,cols,retrieved_at
0,results.csv,https://raw.githubusercontent.com/martj42/inte...,49520,9,2026-08-24T18:54:16.369190
1,fifa_ranking.csv,https://raw.githubusercontent.com/cnc8/fifa-wo...,62424,9,2026-08-24T18:54:16.369202


## Note on the Elo ratings source

`eloratings.net` (World Football Elo Ratings) is not fetched automatically in this notebook —
it needs to be supplied manually as `data/external/elo_ratings.csv` (see `DATA_SOURCES.md` for
mirror links). The feature engineering notebook checks for this file and will use it if present,
or fall back to computing a proxy Elo rating directly from match results if it's absent.
